# Decoder-Only Transformer: Complete Pipeline Lab

## Full Pipeline

### Training
`Text Corpus → Tokenization → Embedding → Positional Encoding → Decoder Stack (Masked Self-Attention + FFN) → Next Token Prediction Head → Loss → Backpropagation → Saved Weights (.pth/.bin/.safetensors)`

### Inference
`User Prompt → Tokenization → Embedding → Positional Encoding → Decoder Stack → Next Token Prediction → Loop Until End Token`

---

**Examples:** GPT, GPT-2, GPT-3, LLaMA, Mistral, Falcon

**Key difference from encoder-only:** Decoder-only models **generate** text autoregressively — they predict the next token based on all previous tokens. Attention is **causal** (masked) — each token can only see tokens before it, never ahead.

We will build each stage one by one.

---
## Step 1: Text Corpus

### What?
Raw, **unlabeled** text data. Decoder-only models learn by predicting the **next token** given all previous tokens — no labels, no pairs, just raw text.

### Why?
- The training objective is **causal language modeling (CLM)**: given tokens `[t1, t2, t3]`, predict `[t2, t3, t4]`.
- The text itself provides the supervision — every token is both input and label (shifted by one position).
- This is why GPT can be trained on massive unlabeled internet text.

### Where in the pipeline?
```
👉 [Text Corpus] → Tokenization → Embedding → PE → Decoder Stack → Next Token Prediction → Loss
```

### Decoder-only vs Encoder-only vs Encoder-Decoder data
| | Decoder-Only | Encoder-Only | Encoder-Decoder |
|--|-------------|-------------|----------------|
| Data format | Raw text | (text, label) | (source, target) pairs |
| Supervision | Self-supervised (next token) | Self-supervised (MLM) or labeled | Paired sequences |
| Example | "The cat sat on the mat" | ("Great movie!", positive) | ("Hello", "Bonjour") |

In [ ]:
# Step 1: Text Corpus — Raw Text for Causal Language Modeling

# Decoder-only models learn from raw text — no labels, no pairs.
# The model learns to predict the next word given previous words.

corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the dog",
    "the dog chased the cat",
    "a bird flew over the mat",
    "the cat and the dog are friends",
    "a bird sat on the rug",
    "the dog and the cat played together",
]

print(f"Corpus size: {len(corpus)} sentences")
print(f"\nTraining objective (Causal LM):")
print(f"  Input:  'the cat sat on'")
print(f"  Target: 'cat sat on the'  ← shifted by 1 position")
print(f"\nEvery token is both input AND label — just shifted!")

### Key Takeaway

- Decoder-only uses **raw unlabeled text** — no pairs, no labels
- Training signal: predict the **next token** from previous tokens
- Input and target are the **same sequence shifted by 1** position

**Next Step →** Tokenization: converting text into token IDs.

---
## Step 2: Tokenization

### What?
Converting raw text into **token IDs**. For decoder-only models, tokenization is simpler than BERT — no `[CLS]` or `[SEP]`. Just the tokens plus an end-of-sequence marker.

### Why?
- The model processes tokens left-to-right, predicting the next one at each step.
- `<eos>` tells the model when a sequence ends (important for generation stopping).
- No `[CLS]` needed — decoder-only doesn't classify, it generates.

### Where in the pipeline?
```
Text Corpus → 👉 [Tokenization] → Embedding → PE → Decoder Stack → Next Token Prediction
```

### Special Tokens
| Token | ID | Purpose |
|-------|----|---------|
| `<pad>` | 0 | Padding shorter sequences |
| `<eos>` | 1 | End of sequence — model learns to stop here |
| `<unk>` | 2 | Unknown / out-of-vocabulary token |

### How input/target are created for Causal LM
```
Original:  [the, cat, sat, on, the, mat, <eos>]
Input:     [the, cat, sat, on, the, mat]        ← everything except last
Target:    [cat, sat, on, the, mat, <eos>]       ← everything except first (shifted by 1)
```
The model at position i predicts the token at position i+1.

In [ ]:
# Step 2: Tokenization — Building Vocabulary & Creating Input/Target Pairs

import torch
import torch.nn as nn
import math

# Special tokens
PAD_TOKEN, EOS_TOKEN, UNK_TOKEN = '<pad>', '<eos>', '<unk>'
PAD_IDX, EOS_IDX, UNK_IDX = 0, 1, 2

class Tokenizer:
    """Word-level tokenizer for decoder-only model."""
    
    def __init__(self):
        self.word2idx = {PAD_TOKEN: 0, EOS_TOKEN: 1, UNK_TOKEN: 2}
        self.idx2word = {v: k for k, v in self.word2idx.items()}
    
    def build_vocab(self, sentences):
        for sent in sentences:
            for word in sent.lower().split():
                if word not in self.word2idx:
                    idx = len(self.word2idx)
                    self.word2idx[word] = idx
                    self.idx2word[idx] = word
        print(f"Vocab size: {len(self.word2idx)}")
    
    def encode(self, sentence):
        return [self.word2idx.get(w, UNK_IDX) for w in sentence.lower().split()]
    
    def decode(self, ids):
        tokens = []
        for i in ids:
            if i == EOS_IDX:
                break
            if i != PAD_IDX:
                tokens.append(self.idx2word.get(i, UNK_TOKEN))
        return ' '.join(tokens)


tokenizer = Tokenizer()
tokenizer.build_vocab(corpus)

# Demonstrate input/target creation for causal LM
text = corpus[0]
full_ids = tokenizer.encode(text) + [EOS_IDX]
input_ids  = full_ids[:-1]   # everything except last
target_ids = full_ids[1:]    # everything except first (shifted by 1)

print(f"\nText:      '{text}'")
print(f"Full IDs:  {full_ids}")
print(f"Input:     {input_ids}  ← model sees this")
print(f"Target:    {target_ids}  ← model predicts this")
print(f"\nAt each position, the model predicts the NEXT token:
")
for i in range(len(input_ids)):
    ctx = tokenizer.decode(input_ids[:i+1])
    tgt = tokenizer.decode([target_ids[i]])
    print(f"  '{ctx}' → predict '{tgt}'")

### Key Takeaway

- Decoder-only tokenization is simple: just tokens + `<eos>` (no `[CLS]`/`[SEP]`)
- Input and target are the **same sequence shifted by 1 position**
- At position `i`, the model predicts token at position `i+1`
- Single vocabulary (same as encoder-only, unlike encoder-decoder)

**Next Step →** Embedding: converting token IDs into dense vectors.

---
## Step 3: Embedding

### What?
A learnable lookup table mapping each token ID to a dense vector of size `d_model`. Identical concept to encoder-only and encoder-decoder — the difference is in what happens **after** embedding (causal masking).

### Why?
- Token IDs are arbitrary integers with no semantic meaning.
- Embeddings place tokens in continuous space where similar words cluster together.
- Single embedding layer (one vocabulary).

### Where in the pipeline?
```
Text Corpus → Tokenization → 👉 [Embedding] → PE → Decoder Stack → Next Token Prediction
```

In [ ]:
# Step 3: Embedding — Token IDs → Dense Vectors

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.d_model = d_model
    
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


d_model = 256
vocab_size = len(tokenizer.word2idx)

embedding = TokenEmbedding(vocab_size, d_model)

input_tensor = torch.tensor([input_ids])  # (1, seq_len)
embedded = embedding(input_tensor)          # (1, seq_len, d_model)

print(f"Vocab size: {vocab_size}")
print(f"Input tensor: {input_tensor} → shape {input_tensor.shape}")
print(f"Embedded shape: {embedded.shape} → (batch, seq_len, d_model)")
print(f"\nFirst token embedding (first 10 dims): {embedded[0, 0, :10].detach()}")

### Key Takeaway

- Embedding is identical across all transformer types — the magic is in the **attention masking**
- Scaled by `√(d_model)` to balance with positional encoding
- Output: `(batch, seq_len, d_model)`

**Next Step →** Positional Encoding: adding position information.

---
## Step 4: Positional Encoding

### What?
Adds position information to embeddings. Same sinusoidal approach as other transformer types.

### Why?
- Transformers process all tokens in parallel — no inherent order.
- Position is critical for language: "the cat chased the dog" ≠ "the dog chased the cat".

### Decoder-only specifics
- Modern GPT-style models often use **learned** positional embeddings or **RoPE** (Rotary Position Embedding).
- We use sinusoidal for simplicity — the concept is the same.
- The key difference isn't in PE — it's in the **causal mask** applied during attention (next step).

### Where in the pipeline?
```
Text Corpus → Tokenization → Embedding → 👉 [Positional Encoding] → Decoder Stack → Next Token Prediction
```

In [ ]:
# Step 4: Positional Encoding

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


pos_encoder = PositionalEncoding(d_model, dropout=0.1)
pos_encoded = pos_encoder(embedded)

print(f"After positional encoding: {pos_encoded.shape}")
print(f"\nBefore PE (first 5 dims): {embedded[0, 0, :5].detach()}")
print(f"After  PE (first 5 dims): {pos_encoded[0, 0, :5].detach()}")

### Key Takeaway

- PE is identical across transformer types — adds position info to embeddings
- The real decoder-only magic happens in the **next step**: causal masked attention

**Next Step →** Decoder Stack: causal self-attention + FFN.

---
## Step 5: Decoder Stack (Masked Self-Attention + FFN)

### What?
A stack of N identical layers, each containing:
1. **Masked (Causal) Multi-Head Self-Attention** — each token can only attend to itself and **previous** tokens
2. **Feed-Forward Network** — position-wise non-linear transformation
3. **Residual connections + Layer Normalization**

### Why causal masking? (THE key concept)
- During generation, token 5 hasn't been generated yet when predicting token 4.
- The causal mask enforces this during training too — so training matches inference.
- Without the mask, the model would "cheat" by looking at future tokens.

### The causal mask visualized
```
Token:    the  cat  sat  on   the  mat
the    [  ✅   ❌   ❌   ❌   ❌   ❌  ]  ← 'the' sees only itself
cat    [  ✅   ✅   ❌   ❌   ❌   ❌  ]  ← 'cat' sees 'the', 'cat'
sat    [  ✅   ✅   ✅   ❌   ❌   ❌  ]  ← 'sat' sees 'the', 'cat', 'sat'
on     [  ✅   ✅   ✅   ✅   ❌   ❌  ]
the    [  ✅   ✅   ✅   ✅   ✅   ❌  ]
mat    [  ✅   ✅   ✅   ✅   ✅   ✅  ]  ← 'mat' sees everything before it
```

### Decoder-only vs Encoder-only vs Encoder-Decoder
| | Decoder-Only | Encoder-Only | Encoder-Decoder (decoder part) |
|--|-------------|-------------|-------------------------------|
| Self-Attention | **Causal** (masked) | **Bidirectional** (no mask) | **Causal** (masked) |
| Cross-Attention | ❌ None | ❌ None | ✅ Attends to encoder output |
| Sub-layers per layer | **2** (self-attn + FFN) | **2** (self-attn + FFN) | **3** (self-attn + cross-attn + FFN) |

### Where in the pipeline?
```
Text Corpus → Tokenization → Embedding → PE → 👉 [Decoder Stack] → Next Token Prediction
```

In [ ]:
# Step 5: Decoder Stack — Causal Self-Attention + FFN

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.W_q(query).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.W_o(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=1024, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)


class DecoderLayer(nn.Module):
    """Single decoder layer: Causal Self-Attention + FFN.
    Only 2 sub-layers (no cross-attention — that's encoder-decoder only)."""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Sub-layer 1: Causal Self-Attention + Residual + Norm
        attn_out = self.self_attn(x, x, x, mask)  # Q=K=V=x, with causal mask
        x = self.norm1(x + self.dropout1(attn_out))
        # Sub-layer 2: FFN + Residual + Norm (NO cross-attention!)
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        return x


class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


def generate_causal_mask(size):
    """Lower-triangular mask: position i can attend to positions 0..i only."""
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return ~mask  # True = allowed, False = blocked


# Config
n_layers = 4
n_heads = 4
d_ff = 1024
dropout = 0.1

decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

# Create causal mask
seq_len = pos_encoded.size(1)
causal_mask = generate_causal_mask(seq_len).unsqueeze(0).unsqueeze(0)

print(f"Causal mask (seq_len={seq_len}):")
print(causal_mask.squeeze().int())
print(f"\n1 = can attend, 0 = blocked (future tokens)")

decoder.eval()
with torch.no_grad():
    decoder_output = decoder(pos_encoded, causal_mask)

print(f"\nDecoder input shape:  {pos_encoded.shape}")
print(f"Decoder output shape: {decoder_output.shape}")
print(f"\nEach position's output only contains info from tokens BEFORE it (causal).")

### Key Takeaway

- Decoder-only has **2 sub-layers** per layer: Causal Self-Attention + FFN
- **No cross-attention** — that's only in encoder-decoder models
- **Causal mask** = lower-triangular matrix — position `i` sees only `0..i`
- This is what makes it autoregressive: each position only knows the past

**Next Step →** Next Token Prediction Head: converting decoder output to vocabulary probabilities.

---
## Step 6: Next Token Prediction Head

### What?
A **linear projection** that maps each decoder output vector (`d_model`) to the **vocabulary size**, producing a score (logit) for every possible next token.

### Why?
- The decoder outputs vectors of size `d_model` — we need to pick a **word**.
- `Linear(d_model → vocab_size)` gives a score for every token in the vocabulary.
- Softmax converts scores to probabilities.

### Decoder-only vs Encoder-only head
| | Decoder-Only | Encoder-Only |
|--|-------------|-------------|
| Head output | `(batch, seq_len, vocab_size)` | `(batch, n_classes)` |
| Predicts at | **Every position** (next token) | **[CLS] only** (class) |
| Projection | `d_model → vocab_size` | `d_model → n_classes` |

### Where in the pipeline?
```
... → Decoder Stack → 👉 [Next Token Prediction Head] → Loss
```

### Weight tying (production optimization)
Many GPT models **share weights** between the embedding layer and the output head. Since both map between `vocab_size` and `d_model`, sharing saves parameters.

In [ ]:
# Step 6: Next Token Prediction Head

class LMHead(nn.Module):
    """Language Model Head: projects decoder output to vocabulary logits."""
    
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.projection = nn.Linear(d_model, vocab_size)
    
    def forward(self, decoder_output):
        # (batch, seq_len, d_model) → (batch, seq_len, vocab_size)
        return self.projection(decoder_output)


lm_head = LMHead(d_model, vocab_size)

with torch.no_grad():
    logits = lm_head(decoder_output)

predicted_ids = logits.argmax(dim=-1)

print(f"Decoder output shape: {decoder_output.shape}")
print(f"Logits shape:         {logits.shape} → (batch, seq_len, vocab_size={vocab_size})")
print(f"\nPredicted next tokens at each position:")
for i in range(len(input_ids)):
    ctx = tokenizer.decode(input_ids[:i+1])
    pred = tokenizer.decode([predicted_ids[0, i].item()])
    actual = tokenizer.decode([target_ids[i]])
    print(f"  '{ctx}' → predicted: '{pred}' | actual: '{actual}'")

### Key Takeaway

- LM Head: `Linear(d_model → vocab_size)` at **every** position
- Predicts the next token at each position in the sequence
- Untrained model gives random predictions — training fixes this

**Next Step →** Loss: measuring how wrong the next-token predictions are.

---
## Step 7: Loss

### What?
**Cross-Entropy Loss** computed at **every position** — comparing the predicted next token vs the actual next token.

### Why?
- At each position `i`, the model predicts what token comes at position `i+1`.
- Loss = average of `-log(probability of correct next token)` across all positions.
- This is the **causal language modeling (CLM)** objective used by GPT.

### Loss comparison across architectures
| | Decoder-Only | Encoder-Only | Encoder-Decoder |
|--|-------------|-------------|----------------|
| Loss at | Every position | [CLS] only | Every target position |
| Predicts | Next token | Class label | Target token |
| Shape | `(batch*seq_len, vocab)` | `(batch, n_classes)` | `(batch*tgt_len, vocab)` |

### Where in the pipeline?
```
... → Decoder Stack → Next Token Prediction Head → 👉 [Loss] → Backpropagation
```

In [ ]:
# Step 7: Loss — Causal Language Modeling Loss

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

target_tensor = torch.tensor([target_ids])  # (1, seq_len)

# Reshape: (batch*seq_len, vocab_size) vs (batch*seq_len)
logits_flat = logits.view(-1, vocab_size)
target_flat = target_tensor.view(-1)

loss = criterion(logits_flat, target_flat)

print(f"Logits shape (flat):  {logits_flat.shape} → (seq_len, vocab_size)")
print(f"Target shape (flat):  {target_flat.shape}  → (seq_len)")
print(f"Target tokens:        {tokenizer.decode(target_flat.tolist())}")
print(f"\nLoss: {loss.item():.4f}")
print(f"\n(High loss expected — untrained model makes random predictions)")

### Key Takeaway

- Loss is computed at **every position** (not just [CLS] like encoder-only)
- `ignore_index=PAD_IDX` ensures padding doesn't affect loss
- This is the standard **causal LM** objective: predict next token at each step

**Next Step →** Backpropagation: training the full model.

---
## Step 8: Backpropagation & Training

### What?
Compute gradients and update weights. Assemble the full decoder-only model and train on the corpus.

### Where in the pipeline?
```
... → Next Token Prediction Head → Loss → 👉 [Backpropagation] → Saved Weights
```

In [ ]:
# Step 8: Backpropagation — Full Training Loop

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, dropout=0.1):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.lm_head = LMHead(d_model, vocab_size)
        # Weight tying: share embedding and output projection weights
        self.lm_head.projection.weight = self.embedding.embedding.weight
    
    def forward(self, x, mask=None):
        x = self.pos_encoding(self.embedding(x))
        x = self.decoder(x, mask)
        return self.lm_head(x)


# Prepare batched data for causal LM
def prepare_batch(corpus, tokenizer):
    all_inputs, all_targets = [], []
    for text in corpus:
        ids = tokenizer.encode(text) + [EOS_IDX]
        all_inputs.append(ids[:-1])   # input: everything except last
        all_targets.append(ids[1:])   # target: shifted by 1
    max_len = max(len(s) for s in all_inputs)
    inputs = torch.tensor([s + [PAD_IDX] * (max_len - len(s)) for s in all_inputs])
    targets = torch.tensor([s + [PAD_IDX] * (max_len - len(s)) for s in all_targets])
    return inputs, targets


# Build model
model = DecoderOnlyTransformer(
    vocab_size=vocab_size, d_model=d_model, n_heads=n_heads,
    n_layers=n_layers, d_ff=d_ff, dropout=dropout
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

input_batch, target_batch = prepare_batch(corpus, tokenizer)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input batch:  {input_batch.shape}")
print(f"Target batch: {target_batch.shape}")

# Training loop
model.train()
n_epochs = 200

for epoch in range(n_epochs):
    causal_mask = generate_causal_mask(input_batch.size(1)).unsqueeze(0).unsqueeze(0)
    
    logits = model(input_batch, causal_mask)
    loss = criterion(logits.view(-1, vocab_size), target_batch.view(-1))
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    if (epoch + 1) % 40 == 0:
        # Calculate accuracy (ignoring padding)
        preds = logits.argmax(dim=-1)
        mask = target_batch != PAD_IDX
        acc = (preds[mask] == target_batch[mask]).float().mean()
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {loss.item():.4f} | Accuracy: {acc.item():.2%}")

print(f"\nTraining complete! Final loss: {loss.item():.4f}")

### Key Takeaway

- Full model: Embedding + PE + Decoder (causal) + LM Head
- **Weight tying**: embedding and LM head share the same weight matrix (saves params)
- Input/target are the same sequence shifted by 1 position
- Causal mask ensures the model can't cheat by looking ahead

**Next Step →** Saved Weights: persisting the trained model.

---
## Step 9: Saved Weights (.pth / .bin / .safetensors)

### Where in the pipeline?
```
... → Loss → Backpropagation → 👉 [Saved Weights]
```

In [ ]:
# Step 9: Saving the Trained Model
import json, os

save_dir = 'trained_decoder_only_model'
os.makedirs(save_dir, exist_ok=True)

# 1. Save weights
torch.save(model.state_dict(), f'{save_dir}/model.pth')

# 2. Save config
config = {
    'vocab_size': vocab_size, 'd_model': d_model, 'n_heads': n_heads,
    'n_layers': n_layers, 'd_ff': d_ff, 'dropout': dropout
}
with open(f'{save_dir}/config.json', 'w') as f:
    json.dump(config, f, indent=2)

# 3. Save tokenizer vocab
with open(f'{save_dir}/vocab.json', 'w') as f:
    json.dump(tokenizer.word2idx, f, indent=2)

print(f"Model saved to '{save_dir}/':")
for f in os.listdir(save_dir):
    size = os.path.getsize(f'{save_dir}/{f}')
    print(f"  {f:20s} ({size:,} bytes)")

### Key Takeaway

- Saved model = **Weights + Config + Vocab**
- This completes the **Training Pipeline**!

---

# 🚀 Inference Pipeline

`User Prompt → Tokenization → Embedding → Positional Encoding → Decoder Stack → Next Token Prediction → Loop Until End Token`

**Next Step →** Inference: loading the model and generating text token by token.

---
## Step 10: Inference — Autoregressive Text Generation

### What?
Load the saved model and generate text **one token at a time** in a loop:
1. Start with a prompt (e.g., "the cat")
2. Feed through model → get next token prediction
3. Append predicted token to the prompt
4. Repeat until `<eos>` or max length

### How inference differs from training
| | Training | Inference |
|--|---------|-----------|
| Input | Full sequence (teacher forcing) | Prompt + generated so far |
| Runs | Once per batch (parallel) | N times (one per token) |
| Target | Known (shifted input) | Unknown (being generated) |

### Where in the pipeline?
```
User Prompt: "the cat"
    │
    ├─→ [the, cat]           → model → predict 'sat'
    ├─→ [the, cat, sat]      → model → predict 'on'
    ├─→ [the, cat, sat, on]  → model → predict 'the'
    └─→ ... until <eos>
```

### Decoding strategies
| Strategy | How | Quality |
|----------|-----|---------|
| **Greedy** | Always pick highest probability token | Deterministic, can be repetitive |
| **Top-k sampling** | Sample from top k tokens | More diverse |
| **Top-p (nucleus)** | Sample from smallest set with cumulative prob ≥ p | Best diversity/quality balance |
| **Temperature** | Scale logits before softmax (higher = more random) | Controls creativity |

In [ ]:
# Step 10: Inference — Load Model & Generate Text

# --- Load saved model ---
with open(f'{save_dir}/config.json') as f:
    cfg = json.load(f)

inf_tokenizer = Tokenizer()
with open(f'{save_dir}/vocab.json') as f:
    inf_tokenizer.word2idx = json.load(f)
    inf_tokenizer.idx2word = {int(v): k for k, v in inf_tokenizer.word2idx.items()}

inf_model = DecoderOnlyTransformer(**cfg)
inf_model.load_state_dict(torch.load(f'{save_dir}/model.pth', weights_only=True))
inf_model.eval()
print(f"Model loaded from '{save_dir}/'")

In [ ]:
# --- Greedy Generation ---

@torch.no_grad()
def generate(model, prompt, tokenizer, max_len=20, temperature=1.0):
    """Generate text autoregressively from a prompt."""
    model.eval()
    ids = tokenizer.encode(prompt)
    
    for _ in range(max_len):
        x = torch.tensor([ids])
        mask = generate_causal_mask(len(ids)).unsqueeze(0).unsqueeze(0)
        logits = model(x, mask)
        
        # Get prediction for the LAST position only
        next_logits = logits[0, -1, :] / temperature
        next_token = next_logits.argmax().item()
        
        if next_token == EOS_IDX:
            break
        ids.append(next_token)
    
    return tokenizer.decode(ids)


# --- Top-k Sampling ---

@torch.no_grad()
def generate_topk(model, prompt, tokenizer, max_len=20, k=5, temperature=1.0):
    """Generate text with top-k sampling for diversity."""
    model.eval()
    ids = tokenizer.encode(prompt)
    
    for _ in range(max_len):
        x = torch.tensor([ids])
        mask = generate_causal_mask(len(ids)).unsqueeze(0).unsqueeze(0)
        logits = model(x, mask)
        
        next_logits = logits[0, -1, :] / temperature
        # Keep only top-k, set rest to -inf
        topk_vals, topk_idx = torch.topk(next_logits, k)
        probs = torch.softmax(topk_vals, dim=-1)
        chosen = topk_idx[torch.multinomial(probs, 1)].item()
        
        if chosen == EOS_IDX:
            break
        ids.append(chosen)
    
    return tokenizer.decode(ids)


# Test generation
print('=== Greedy Generation ===\n')
prompts = ['the cat', 'the dog', 'a bird', 'the']
for prompt in prompts:
    result = generate(inf_model, prompt, inf_tokenizer)
    print(f"  '{prompt}' → '{result}'")

print('\n=== Top-k Sampling (k=3, more diverse) ===\n')
for prompt in prompts:
    results = [generate_topk(inf_model, prompt, inf_tokenizer, k=3) for _ in range(3)]
    print(f"  '{prompt}' →")
    for r in results:
        print(f"    '{r}'")

### Key Takeaway

- Inference is **autoregressive**: generate one token, append, repeat
- Model runs **N times** (once per generated token) — unlike encoder-only which runs once
- **Greedy**: deterministic, always picks highest prob token
- **Top-k sampling**: picks from top k tokens — adds diversity/creativity
- **Temperature**: higher = more random, lower = more focused

---

## 🎯 Summary: Complete Decoder-Only Pipeline

### Training Pipeline
```
Text Corpus                 → Raw unlabeled text
  → Tokenization            → tokens + <eos>, input/target shifted by 1
  → Embedding               → Token IDs → dense vectors (d_model)
  → Positional Encoding     → Add position info
  → Decoder Stack           → CAUSAL self-attention + FFN (N layers)
  → Next Token Pred Head    → Linear(d_model → vocab_size) at every position
  → Loss                    → CrossEntropy(predicted vs actual next token)
  → Backpropagation         → Gradients → optimizer → update weights
  → Saved Weights           → .pth + config.json + vocab
```

### Inference Pipeline
```
User Prompt                 → Starting text
  → Tokenization            → Token IDs
  → Embedding + PE          → Dense vectors with position
  → Decoder Stack           → Causal attention (sees only past)
  → Next Token Prediction   → Pick next token (greedy/top-k/top-p)
  → Loop Until <eos>        → Append token, repeat
```

### Key Differences from Encoder-Only & Encoder-Decoder
| Feature | Decoder-Only | Encoder-Only | Encoder-Decoder |
|---------|-------------|-------------|----------------|
| Attention | **Causal** (masked) | Bidirectional | Both |
| Task | **Generation** | Understanding | Seq-to-Seq |
| Inference | **Autoregressive loop** | Single pass | Encode once + decode loop |
| Output | Generated text | Embeddings / classes | Generated text |
| Sub-layers | **2** (self-attn + FFN) | 2 (self-attn + FFN) | 3 (self + cross + FFN) |
| Cross-attention | **❌ No** | ❌ No | ✅ Yes |
| Examples | GPT, LLaMA, Mistral | BERT, RoBERTa | T5, BART |